
## Circuito esquemático

Se optó por la herramienta KiCad v10.0 por tratarse de un entorno de código abierto y por constituir el estándar en el que se encuentran documentados los esquemáticos oficiales de la plataforma **EDU-CIAA-NXP**, facilitando la correspondencia directa de los conectores de expansión **P1** y **P2**.

El plano eléctrico integra los tres bloques funcionales del hardware:
* **Bloque de mando:** joysticks analógicos y pulsador de la pinza.
* **Bloque de procesamiento:** conectores de la placa EDU-CIAA-NXP.
* **Bloque de potencia y actuadores:** servomotores y fuente de alimentación externa.

![Circuito esquemático](esquemático.jpg)

### Figura 1. Circuito esquemático general de conexiones de la EDU-CIAA-NXP y periféricos

---

Para garantizar la compatibilidad con las especificaciones del microcontrolador LPC4337 y preservar la integridad de las entradas analógicas (cuyo fondo de escala es de 3,3 V), los potenciómetros de mando se alimentan directamente desde la rama lógica de la placa. Las consignas de control para los servomotores se generan mediante canales de temporización y modulación por ancho de pulso, adoptando la distribución nativa del módulo `sapi_servo` de la biblioteca estándar **sAPI** para facilitar la integración directa del firmware sin requerir remapeos en la capa de abstracción de periféricos. La distribución adoptada se resume a continuación:

* **Conector P1 (Entradas analógicas, alimentación lógica y control PWM de servos 1 a 4):**
  * **Pin 1 (`+3.3V`):** Alimentación de los módulos de joystick A y B.
  * **Pin 2 (`ANALOG_GND`):** Retorno de masa de referencia para las entradas analógicas.
  * **Pin 3 (`ADC0_3`):** Entrada analógica para el eje X del Joystick B (control de flexión del codo).
  * **Pin 5 (`ADC0_2`):** Entrada analógica para el eje Y del Joystick A (control de elevación del hombro).
  * **Pin 7 (`ADC0_1`):** Entrada analógica para el eje X del Joystick A (control de giro de la base).
  * **Pin 9 (`DAC / ADC0_0`):** Entrada analógica multiplexada para el eje Y del Joystick B (orientación de la muñeca).
  * **Pin 35 (`T_FIL3` / `SERVO3`):** Salida PWM de 50 Hz asignada al servomotor de la Muñeca (`PWM_MUÑECA`).
  * **Pin 36 (`T_FIL1` / `SERVO0`):** Salida PWM de 50 Hz asignada al servomotor de la Base (`PWM_BASE`).
  * **Pin 37 (`T_FIL2` / `SERVO2`):** Salida PWM de 50 Hz asignada al servomotor del Codo (`PWM_CODO`).
  * **Pin 39 (`T_COL0` / `SERVO1`):** Salida PWM de 50 Hz asignada al servomotor del Hombro (`PWM_HOMBRO`).

* **Conector P2 (Entrada digital de pinza, masa y control PWM de servo 5):**
  * **Pin 29 (`GPIO_GND`):** Unión de referencia al plano de masa común de potencia y lógica.
  * **Pin 31 (`GPIO0`):** Entrada digital configurada con resistencia de pull-up interna para el pulsador de apertura/cierre de la pinza (`PINZA_BTN`).
  * **Pin 40 (`GPIO8` / `SERVO4`):** Salida PWM de 50 Hz asignada al servomotor de la Pinza (`PWM_PINZA`).

---
#### Consideraciones de compatibilidad con la biblioteca de firmware (sAPI)

Para simplificar la futura integración del software y prescindir de reconfiguraciones de bajo nivel en las capas de abstracción de hardware, la asignación de pines para el control de los servomotores se realizó respetando de manera estricta el mapa de periféricos nativo provisto por el módulo `sapi_servo` de la biblioteca estándar **sAPI**. 

Dicho controlador implementa la generación de señales PWM para hasta 9 canales preasignados mediante temporizadores e interrupciones del microcontrolador LPC4337, asignando líneas específicas de los conectores P1 y P2 de la EDU-CIAA-NXP. Siguiendo este criterio, se adoptaron los canales `SERVO0` a `SERVO3` sobre la tira P1 y `SERVO4` sobre la tira P2 para gobernar las cinco articulaciones del manipulador robótico, garantizando plena coherencia entre el plano esquemático y las funciones nativas de inicialización y posicionamiento (`servoInit()` y `servoWrite()`).

En la Figura X se ilustra la distribución oficial de pines asignados por sAPI para el control de servomotores en la placa EDU-CIAA-NXP, tomada como referencia para el conexionado del presente diseño:

![Mapeo oficial de pines para servomotores en EDU-CIAA-NXP mediante sAPI](codigo/examples/c/sapi/servo/servo_pins.png)

*Figura 2: Mapeo de salidas de control para servomotores (SERVO0 a SERVO8) en los conectores P1 y P2 de la EDU-CIAA-NXP bajo la biblioteca sAPI.*


---
#### Código de conexión de los servomotores (MG996R)

Para el conexionado físico entre las tiras de pines del esquemático (`J_SERVO1` a `J_SERVO5`) y los cables de los servomotores MG996R, se adopta el estándar comercial habitual (cable plano de 3 vías con ficha Dupont hembra de paso 2,54 mm):

* **Pin 1 — Masa (`GND`):** Conductor de color **marrón** (o negro). Retorno de referencia común a la masa del sistema.
* **Pin 2 — Alimentación (`+6V`):** Conductor de color **rojo**. Conexión directa a la línea protegida de 6 V proveniente de la fuente externa.
* **Pin 3 — Control (`PWM`):** Conductor de color **naranja** (o amarillo). Entrada de la señal digital modulada generada por las salidas de la tira P2 de la EDU-CIAA.